### RAG Pipelines- Data Ingestion to Vector Database Pipelines


In [75]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [76]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

def process_all_pdfs(directory_path):
    """Process all PDF files in the given directory."""
    
    all_documents = []
    pdf_dir = Path(directory_path)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

        print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents


all_pdf_documents = process_all_pdfs("../data/pdf")

Found 6 PDF files to process

Processing: JD.pdf
Loaded 2 pages

Total documents loaded: 2

Processing: Multi Task Learning Framework.pdf
Loaded 12 pages

Total documents loaded: 14

Processing: RAG.pdf
Loaded 21 pages

Total documents loaded: 35

Processing: Ress1.pdf
Loaded 2 pages

Total documents loaded: 37

Processing: Ress2.pdf
Loaded 1 pages

Total documents loaded: 38

Processing: Ress3.pdf
Loaded 2 pages

Total documents loaded: 40


In [77]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-07-10T13:32:19+05:30', 'source': '..\\data\\pdf\\JD.pdf', 'file_path': '..\\data\\pdf\\JD.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Akshay Gupta', 'subject': '', 'keywords': '', 'moddate': '2026-07-10T13:32:19+05:30', 'trapped': '', 'modDate': "D:20260710133219+05'30'", 'creationDate': "D:20260710133219+05'30'", 'page': 0, 'source_file': 'JD.pdf', 'file_type': 'pdf'}, page_content='Job Description: Lead .NET developer (5-8 yrs) \n1. Project Overview \nJoin our engineering team developing digital manufacturing solutions for a leading Smart Meter \nmanufacturing company. The project focuses on production traceability, quality management, \nmanufacturing execution, real-time dashboards, shop-floor digitization, and enterprise application \ndevelopment. You will work closely with production, quality, IT, and business teams to 

In [78]:
### Text splitting get into chuncks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into chunks for good performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", " ", ""]
        )
    split_docs = text_splitter.split_documents(documents)
    print(f"Splitted {len(documents)} documents from {len(split_docs)} chuncks")

    # Show Example of a chunk
    if split_docs:
        print("\nExample of a chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}..")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [79]:
chunks=split_documents(all_pdf_documents)

Splitted 40 documents from 222 chuncks

Example of a chunk:
Content: Job Description: Lead .NET developer (5-8 yrs) 
1. Project Overview 
Join our engineering team developing digital manufacturing solutions for a leading Smart Meter 
manufacturing company. The project ..
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-07-10T13:32:19+05:30', 'source': '..\\data\\pdf\\JD.pdf', 'file_path': '..\\data\\pdf\\JD.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Akshay Gupta', 'subject': '', 'keywords': '', 'moddate': '2026-07-10T13:32:19+05:30', 'trapped': '', 'modDate': "D:20260710133219+05'30'", 'creationDate': "D:20260710133219+05'30'", 'page': 0, 'source_file': 'JD.pdf', 'file_type': 'pdf'}


### Embedding and VectorStoreDB

In [80]:
import uuid
from typing import Any, Dict, List, Tuple

import chromadb
from chromadb.config import Settings
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [81]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.

        Args:
            model_name (str): HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model."""
        try:
            print(f"Loading Sentence Transformer model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts:List[str])-> np.ndarray:
        """Generate embeddings for a list of texts using the Sentence Transformer model

        Args:
        texts : List of texts strings to embed

        Returns:
        numpy array of embeddings with shape (len(texts), wmbedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded. Please call _load_model() first.")

        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts,batch_size=64,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading Sentence Transformer model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5677.66it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\109899\AppData\Local\Temp\ipykernel_23112\3466932639.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


### VectorStore



In [82]:
import os
import uuid
from typing import List
import numpy as np
import chromadb
# Assuming LangChain's Document schema or similar
from langchain_core.documents import Document  


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """Initialize the vector store

        Args:
            collection_name (str): Name of the ChromaDB collection
            persist_directory (str): Path to directory where vector store will be persisted
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB Client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or Create Collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(
                f"Vector store initialized successfully. Collection: {self.collection_name}"
            )
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:  # Fixed indentation here
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store

        Args:
            documents (List[Document]): List of Document objects to add
            embeddings (np.ndarray): Embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")

        print(f"Adding {len(documents)} documents to vector store")

        # Fixed comment syntax here (# instead of @)
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID (Fixed uuid.uuid4 typo here)
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_texts.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts,
            )
            print(
                f"Successfully added {len(documents)} documents to vector store"
            )
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Instantiation placed outside the class scope
if __name__ == "__main__":
    vectorstore = VectorStore()

Vector store initialized successfully. Collection: pdf_documents
Existing documents in collection: 622


In [83]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-07-10T13:32:19+05:30', 'source': '..\\data\\pdf\\JD.pdf', 'file_path': '..\\data\\pdf\\JD.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Akshay Gupta', 'subject': '', 'keywords': '', 'moddate': '2026-07-10T13:32:19+05:30', 'trapped': '', 'modDate': "D:20260710133219+05'30'", 'creationDate': "D:20260710133219+05'30'", 'page': 0, 'source_file': 'JD.pdf', 'file_type': 'pdf'}, page_content='Job Description: Lead .NET developer (5-8 yrs) \n1. Project Overview \nJoin our engineering team developing digital manufacturing solutions for a leading Smart Meter \nmanufacturing company. The project focuses on production traceability, quality management, \nmanufacturing execution, real-time dashboards, shop-floor digitization, and enterprise application \ndevelopment. You will work closely with production, quality, IT, and business teams to 

In [84]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 222 texts


Batches: 100%|██████████| 4/4 [00:10<00:00,  2.51s/it]


Generated embeddings with shape: (222, 384)
Adding 222 documents to vector store
Successfully added 222 documents to vector store
Total documents in collection: 844


Retriver Pipleine From VectorStore

In [85]:
from typing import List, Dict, Any
import numpy as np

class VectorRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: Any, embedding_manager: Any):
        """
        Initialize the retriever

        Args:
            vector_store: Vector Store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve Relevant documents for a query

        Args:
            query: The search query
            top_k: Number of results to return
            score_threshold: Minimum score for results

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate Query Embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Ensure query_embedding is converted to a list safely
        if hasattr(query_embedding, "tolist"):
            query_embedding = query_embedding.tolist()

        # Search query Embedding
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )

            # Process results
            retrieved_docs = []

            if results and results.get('documents') and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1,
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents retrieved or results are empty")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

# Corrected instantiation using VectorRetriever
rag_retriever = VectorRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)

In [86]:
rag_retriever

In [87]:
rag_retriever.retrieve("What kind of RAG is you need")

Retrieving documents for query: 'What kind of RAG is you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.40it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bfb2dbe3_8',
  'content': 'This survey endeavors to\nfill this gap by mapping out the RAG process and charting\nits evolution and anticipated future paths, with a focus on the\nintegration of RAG within LLMs. This paper considers both\ntechnical paradigms and research methods, summarizing three\nmain research paradigms from over 100 RAG studies, and\nanalyzing key technologies in the core stages of “Retrieval,”\n“Generation,” and “Augmentation.” On the other hand, current\nresearch tends to focus more on methods, lacking analysis and\nsummarization of how to evaluate RAG. This paper compre-\nhensively reviews the downstream tasks, datasets, benchmarks,\nand evaluation methods applicable to RAG. Overall, this\npaper sets out to meticulously compile and categorize the\nfoundational technical concepts, historical progression, and\nthe spectrum of RAG methodologies and applications that\nhave emerged post-LLMs. It is designed to equip readers and\nprofessionals with a detailed

In [98]:
rag_retriever.retrieve("RAG")

Retrieving documents for query: 'RAG'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bfb2dbe3_8',
  'content': 'This survey endeavors to\nfill this gap by mapping out the RAG process and charting\nits evolution and anticipated future paths, with a focus on the\nintegration of RAG within LLMs. This paper considers both\ntechnical paradigms and research methods, summarizing three\nmain research paradigms from over 100 RAG studies, and\nanalyzing key technologies in the core stages of “Retrieval,”\n“Generation,” and “Augmentation.” On the other hand, current\nresearch tends to focus more on methods, lacking analysis and\nsummarization of how to evaluate RAG. This paper compre-\nhensively reviews the downstream tasks, datasets, benchmarks,\nand evaluation methods applicable to RAG. Overall, this\npaper sets out to meticulously compile and categorize the\nfoundational technical concepts, historical progression, and\nthe spectrum of RAG methodologies and applications that\nhave emerged post-LLMs. It is designed to equip readers and\nprofessionals with a detailed

Integration Vectordb Context Pipeline With LLM Output

In [89]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

# 1. Initialize the LLM with an active, supported model name
llm = ChatGroq(
    model="openai/gpt-oss-20b",  # llama-3.3-70b-versatile is enterprise-only
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# 2. Define the corrected RAG function
def rag_simple(query, retriever, llm, top_k=3):
    # Custom VectorRetriever uses retrieve() and returns dicts with 'content'
    docs = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc["content"] for doc in docs)

    # Construct prompt
    prompt = f"""Use the following context to answer the question.

    Context:
    {context}

    Question: {query}

    Answer:"""

    # Invoke LLM
    response = llm.invoke(prompt)

    return response.content



In [97]:
answer=rag_simple(" RAG",rag_retriever,llm)
print(answer)

Retrieving documents for query: ' RAG'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**RAG (Retrieval‑Augmented Generation)**  
RAG is a research paradigm that blends *retrieval* of relevant external knowledge with *generation* by large language models (LLMs). The goal is to produce more accurate, grounded, and context‑aware responses by feeding the model with up‑to‑date or domain‑specific documents during inference.

Key components highlighted in the survey:

| Stage | What it does | Typical techniques |
|-------|--------------|--------------------|
| **Retrieval** | Finds documents or passages that are relevant to the user query. | Dense vector search (e.g., FAISS), sparse BM25, hybrid retrieval. |
| **Generation** | The LLM generates the final answer, conditioned on the retrieved content. | Prompt‑engineering, fine‑tuning, retrieval‑augmented decoding. |
| **Augmentation** | Enhances the retrieved material (e.g., summarization, paraphrasing) before feeding it to the generator. | Summarization models, relevance re‑ranking, knowledge distillation. |

**Research Landsc

Enhanced RAG Pipeline Features

In [96]:
from langchain_core.messages import HumanMessage

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG Pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # Clean whitespace from the input query
    clean_query = query.strip()

    # 1. Retrieve relevant documents
    results = retriever.retrieve(clean_query, top_k=top_k, score_threshold=min_score)
    if not results:
        empty_response = {
            'answer': 'No relevant Context Found.',
            'sources': [],
            'confidence': 0.0
        }
        if return_context:
            empty_response['context'] = ''
        return empty_response

    # 2. Prepare context, sources, and confidence
    context = "\n\n".join([doc['content'] for doc in results])
    
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    
    confidence = max([doc['similarity_score'] for doc in results])

    # 3. Construct prompt using direct f-string formatting
    prompt = f"Use the following context to answer the question concisely.\n\nContext:\n{context}\n\nQuestion: {clean_query}\n\nAnswer:"

    # 4. Invoke LLM safely
    response = llm.invoke(prompt)
    answer_text = response.content if hasattr(response, 'content') else str(response)

    # 5. Format and return output
    output = {
        'answer': answer_text,
        'sources': sources,
        'confidence': confidence
    }
    
    if return_context:
        output['context'] = context
        
    return output


# --- Example Usage ---
# Make sure 'rag_retriever' and 'llm' are initialized beforehand with valid parameters/models
result = rag_advanced("RAG", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)

print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result.get('context', '')[:300])

Retrieving documents for query: 'RAG'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.30it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: **RAG** – Retrieval‑Augmentation‑Generation, a framework that combines a retrieval step (fetching relevant documents), an augmentation step (integrating retrieved content into the model’s context), and a generation step (producing the final output) to enhance large language models.
Sources: [{'source': 'RAG.pdf', 'page': 0, 'score': 0.23140251636505127, 'preview': 'This survey endeavors to\nfill this gap by mapping out the RAG process and charting\nits evolution and anticipated future paths, with a focus on the\nintegration of RAG within LLMs. This paper considers both\ntechnical paradigms and research methods, summarizing three\nmain research paradigms from over 1...'}, {'source': 'RAG.pdf', 'page': 0, 'score': 0.23140251636505127, 'preview': 'This survey endeavors to\nfill this gap by mapping out the RAG process and charting\nits evolution and anticipated future paths, with a focus on the\ninteg

In [95]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is RAG?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is RAG?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
the user’s query. These articles, combined with the original
question, form a comprehensive prompt that empowers LLMs
to generate a well-informed answer.
The RAG research 

paradigm is continuously evolving, and
we categorize it into three stages: Naive RAG, Advanced
RAG, and Modular RAG, as showed in Figure 3. Despite
RAG method are cost-effective and surpass the performance
of the native LLM, they also exhibit several limitations.
The development of Advanced RAG and Modular RAG is
a response to these specific shortcomings in Naive RAG.
A. Naive RAG
The Naive RAG research paradigm represents the earli-
est methodology, which gained prominence shortly after the

the user’s query. These articles, combined with the original
question, form a comprehensive prompt that empowers LLMs
to generate a well-informed answer.
The RAG research paradigm is continuously evolving, and
we categorize it into three stages: Naive RAG, Advanced
RAG, and Modular RAG, as showed in Figure 3. Despite
RAG method are cost-effective and surpass the performance
of the native LLM, they also exhibit several limitations.
The development of Advanced RAG and Modular RAG is
a response to th